# Mobile Price Classification — Exploratory Data Analysis / EDA (Tahap 2)
        **Praktikum 14.16 | CRISP-DM | Dataset Kaggle**

        **Tujuan:** Menganalisis statistik, korelasi, distribusi, dan kandidat outlier.

        **Batasan:** belum training 3 algoritma, belum preprocessing, dan belum tuning. Tahap-tahap tersebut dikerjakan kemudian, secara berurutan.

## Cara menjalankan di Google Colab
        1. Buka [Google Colab](https://colab.research.google.com/), pilih **File → Upload notebook** dan unggah file `.ipynb` ini.
        2. Jalankan sel satu per satu dari atas ke bawah (**Shift + Enter**).
        3. Saat diminta, unggah **`train.csv`** dari dataset Kaggle *Mobile Price Classification*; `test.csv` belum diperlukan.
        4. Grafik tampil di bawah sel dan otomatis tersimpan ke folder `figures/`.

        Jika `train.csv` sudah ada di folder notebook, dialog upload tidak akan muncul.
**Catatan:** Di Colab, grafik tampil langsung di bawah sel dan dapat dilihat dengan scroll. Tombol **Previous / Next** tersedia di file `.py` yang dijalankan dari terminal Windows.


## Memuat dataset `train.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Dapat dijalankan di Google Colab maupun Jupyter lokal.
# Jika train.csv belum ditemukan, Google Colab akan meminta upload.
calon_file = [
    Path('Mobile_Price_Classification/train.csv'),
    Path('train.csv'),
    Path('/content/train.csv'),
]
file_csv = next((p for p in calon_file if p.is_file()), None)

if file_csv is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'train.csv belum ditemukan. Taruh di folder notebook atau '
            'di folder Mobile_Price_Classification/'
        ) from exc
    print('Silakan upload train.csv dari dataset Kaggle Mobile Price Classification.')
    diupload = files.upload()
    if 'train.csv' not in diupload:
        raise FileNotFoundError('File yang diupload harus bernama train.csv.')
    file_csv = Path('train.csv')

df = pd.read_csv(file_csv)
if 'price_range' not in df.columns:
    raise ValueError('Kolom target price_range tidak ditemukan. Gunakan train.csv, bukan test.csv.')

print('Dataset:', file_csv)
print('Ukuran:', df.shape)

## 1. Statistik deskriptif

In [ ]:
statistik = df.describe().T.round(2)
print('=== STATISTIK DESKRIPTIF ===')
display(statistik)

## 2. Korelasi fitur terhadap target

In [ ]:
korelasi = df.corr(numeric_only=True)
korelasi_target = korelasi['price_range'].drop('price_range').sort_values(key=lambda s: s.abs(), ascending=False)
print('=== KORELASI FITUR DENGAN PRICE_RANGE (URUTAN KEKUATAN ABSOLUT) ===')
display(korelasi_target.round(3).rename('Korelasi').to_frame())

## 3. Visualisasi: fitur dengan korelasi terkuat

In [ ]:
OUTPUT = Path('figures/02_eda')
OUTPUT.mkdir(parents=True, exist_ok=True)

sepuluh_teratas = korelasi_target.head(10).sort_values()
fig, ax = plt.subplots(figsize=(9, 5.5))
bar = ax.barh(sepuluh_teratas.index, sepuluh_teratas.values)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlim(-1, 1)
ax.set_xlabel('Koefisien korelasi Pearson dengan price_range')
ax.set_title('10 Fitur dengan Hubungan Terkuat ke Target')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(OUTPUT / '01_korelasi_target.png', dpi=160, bbox_inches='tight')
plt.show()

## 4. Visualisasi: heatmap korelasi fitur

In [ ]:
# Untuk menjaga keterbacaan, ditampilkan 10 fitur paling terkait target + target.
kolom_heatmap = korelasi_target.head(10).index.tolist() + ['price_range']
matriks = df[kolom_heatmap].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10.5, 8.5))
im = ax.imshow(matriks.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(kolom_heatmap)), labels=kolom_heatmap, rotation=60, ha='right')
ax.set_yticks(np.arange(len(kolom_heatmap)), labels=kolom_heatmap)
for i in range(len(kolom_heatmap)):
    for j in range(len(kolom_heatmap)):
        value = matriks.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(value) > 0.65 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8, label='Korelasi Pearson')
ax.set_title('Heatmap Korelasi 10 Fitur Teratas dan Target', pad=14)
fig.tight_layout()
fig.savefig(OUTPUT / '02_heatmap_korelasi.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Visualisasi: boxplot RAM menurut kategori harga

In [ ]:
kelas = sorted(df['price_range'].dropna().unique())
grup_ram = [df.loc[df['price_range'] == kelas_i, 'ram'].dropna() for kelas_i in kelas]
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.boxplot(grup_ram, tick_labels=[str(int(k)) for k in kelas], patch_artist=True,
           boxprops={'facecolor': 'lightsteelblue'})
ax.set_title('Perbandingan RAM pada Tiap Kategori Harga')
ax.set_xlabel('Price range (0 = terendah, 3 = tertinggi)')
ax.set_ylabel('RAM (MB)')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / '03_boxplot_ram.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. Visualisasi: distribusi battery_power menurut kelas

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for k in kelas:
    sub = df.loc[df['price_range'] == k, 'battery_power'].dropna()
    ax.hist(sub, bins=22, alpha=0.45, label=f'Kelas {int(k)}')
ax.set_title('Distribusi Battery Power Menurut Kategori Harga')
ax.set_xlabel('Battery power (mAh)')
ax.set_ylabel('Frekuensi')
ax.legend()
ax.grid(axis='y', alpha=0.18)
fig.tight_layout()
fig.savefig(OUTPUT / '04_histogram_battery.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Deteksi outlier menggunakan IQR (belum dihapus)

In [ ]:
# Tahap EDA hanya mendeteksi outlier, bukan menghapus atau mengganti nilainya.
fitur_cek = ['ram', 'battery_power', 'px_height', 'px_width', 'mobile_wt']
hasil_iqr = []
for fitur in fitur_cek:
    q1, q3 = df[fitur].quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outlier = int(((df[fitur] < bawah) | (df[fitur] > atas)).sum())
    hasil_iqr.append({'Fitur': fitur, 'Batas bawah': bawah,
                      'Batas atas': atas, 'Jumlah outlier IQR': n_outlier})
tabel_iqr = pd.DataFrame(hasil_iqr)
display(tabel_iqr.round(2))
print('Catatan: outlier statistik belum tentu kesalahan data. Keputusan penanganan dilakukan pada tahap preprocessing.')

## 8. Ringkasan EDA untuk laporan ilmiah

In [ ]:
fitur_paling_terkait = korelasi_target.index[0]
nilai_korelasi = korelasi_target.iloc[0]
print('RINGKASAN EXPLORATORY DATA ANALYSIS')
print(f'- Fitur dengan korelasi Pearson absolut tertinggi ke target: {fitur_paling_terkait} ({nilai_korelasi:.3f}).')
print('- Grafik RAM per kategori membantu melihat pola perbedaan spesifikasi antar kelas.')
print('- Analisis IQR digunakan untuk mengidentifikasi kandidat outlier, bukan langsung membuang data.')
print('- Korelasi hanya menunjukkan hubungan statistik, bukan sebab-akibat.')
print(f'- Grafik tersimpan di: {OUTPUT}')

### Catatan untuk laporan ilmiah
        Simpan hasil tabel dan grafik. Pada laporan, **jelaskan angka yang benar-benar muncul saat notebook dijalankan**. Jika memakai data lain atau preprocessing berbeda, hasil juga dapat berubah.

        Jangan menyimpulkan bahwa korelasi membuktikan sebab-akibat atau menyebut data dihapus jika baru dideteksi sebagai outlier.

## Opsional — Unduh semua grafik dari Google Colab

In [ ]:
from pathlib import Path
from zipfile import ZipFile
folder_grafik = Path('figures')
file_zip = Path('grafik_mobile_price_tahap_' + '2' + '.zip')
# Kode ini membuat arsip semua grafik yang ada; tahap 1/2 tidak saling tergantung.
with ZipFile(file_zip, 'w') as arsip:
    for gambar in folder_grafik.rglob('*.png'):
        arsip.write(gambar, gambar.as_posix())
print('Arsip grafik:', file_zip)
try:
    from google.colab import files
    files.download(str(file_zip))
except ImportError:
    print('Anda menjalankan notebook di luar Colab, unduh berkas zip secara manual.')